# Sanity Check: candidates_deduped

Basic validation and inspection of the deduplicated candidates dataset.

In [47]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_parquet('../data/processed/candidates_deduped.parquet')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded: 6,866 rows × 11 columns


## 1. Schema & Data Types

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6866 entries, 0 to 6865
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   problem_id   6866 non-null   object
 1   problem      6866 non-null   object
 2   answer       6866 non-null   object
 3   answer_type  6866 non-null   object
 4   source       6866 non-null   object
 5   difficulty   6866 non-null   object
 6   split        6866 non-null   object
 7   unit         6866 non-null   object
 8   tolerance    6866 non-null   object
 9   domain       6866 non-null   object
 10  language     6866 non-null   object
dtypes: object(11)
memory usage: 590.2+ KB


In [49]:
# Display dtypes more clearly
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

problem_id     object
problem        object
answer         object
answer_type    object
source         object
difficulty     object
split          object
unit           object
tolerance      object
domain         object
language       object
dtype: object

Memory usage: 7.50 MB


## 2. Nulls & Missing Values

In [50]:
null_counts = df.isnull().sum()
null_pct = 100 * df.isnull().sum() / len(df)

null_summary = pd.DataFrame({
    'column': null_counts.index,
    'null_count': null_counts.values,
    'null_pct': null_pct.values
}).sort_values('null_count', ascending=False)

print(null_summary[null_summary['null_count'] > 0])
if (null_summary['null_count'] == 0).all():
    print("✓ No nulls found!")

Empty DataFrame
Columns: [column, null_count, null_pct]
Index: []
✓ No nulls found!


## 3. Duplicates

In [51]:
dup_count = df.duplicated().sum()
dup_pct = 100 * dup_count / len(df)

print(f"Exact duplicates (all columns): {dup_count} ({dup_pct:.2f}%)")

if dup_count > 0:
    print("\nSample duplicate rows:")
    dup_rows = df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10)
    print(dup_rows)

Exact duplicates (all columns): 0 (0.00%)


## 4. Basic Statistics

In [52]:
df.describe(include='all').T

,count,unique,top,freq
problem_id,6866,6866,SciBench_RL_00426,1
problem,6866,6866,Suppose that a fair $n$-sided die is rolled $n...,1
answer,6866,5970,\boxed{0},98
answer_type,6866,87,numerical,2698
source,6866,5,UGPhysics,5451
difficulty,6866,5,,6061
split,6866,1,train_candidate,6866
unit,6866,828,,4761
tolerance,6866,40,,6753
domain,6866,29,QuantumMechanics,999


## 5. Column-Specific Checks

In [53]:
# String columns: check for obvious issues
for col in df.select_dtypes(include=['object']).columns:
    print(f"\n{col}:")
    print(f"  Unique values: {df[col].nunique():,}")
    print(f"  Sample values: {df[col].unique()[:5]}")
    # Check for empty strings
    empty = (df[col] == '').sum()
    if empty > 0:
        print(f"  ⚠ Empty strings: {empty}")


problem_id:
  Unique values: 6,866
  Sample values: ['PHYSICS_00001' 'PHYSICS_00003' 'PHYSICS_00005' 'PHYSICS_00007'
 'PHYSICS_00009']

problem:
  Unique values: 6,866
  Sample values: ['The process of vaporization. At a pressure of $1.013 \\times 10^{5} \\mathrm{~Pa}$, how much does the internal energy of 1 mol of water increase when it turns into steam at $100^{\\circ} \\mathrm{C}$? It is known that at this pressure and temperature, the molar volumes of water and steam are $V_{1, \\mathrm{~m}}=18.8 \\mathrm{~cm}^{3} / \\mathrm{mol}$ and $V_{\\phi, \\mathrm{m}}=3.01 \\times 10^{4} \\mathrm{~cm}^{3} / \\mathrm{mol}$, respectively, and the latent heat of vaporization of water is $L=4.06 \\times 10^{4} \\mathrm{~J} / \\mathrm{mol}$.'
 'Entropy change of hot water. Place 1 kg of water at 20°C on a stove at 100°C to heat it, finally reaching 100°C. The specific heat of water is 4.18 × 10³ J/(kg · K). Calculate the entropy change of the water and the stove, ΔS_w and ΔS_l, respectively.'
 '

## 6. Sample Records (First 10)

In [54]:
df.head(10)

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
0,PHYSICS_00001,The process of vaporization. At a pressure of ...,\boxed{3.75 \times 10^{4}},numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
1,PHYSICS_00003,Entropy change of hot water. Place 1 kg of wat...,"[""\\boxed{1.01 \\times 10^{3}}"", ""\\boxed{-9.0...","[""numerical"", ""numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
2,PHYSICS_00005,Use white light as the light source to observe...,\boxed{1},numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
3,PHYSICS_00007,"Under normal brightness, the diameter of the h...","[""\\boxed{2.24 \\times 10^{-4}}"", ""\\boxed{8.9}""]","[""numerical"", ""numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
4,PHYSICS_00009,Example 25.7: If an object is placed within th...,"[""\\boxed{-60}"", ""\\boxed{4}""]","[""numerical"", ""numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
5,PHYSICS_00011,Calculate the de Broglie wavelength of a bulle...,\boxed{2.21 \times 10^{-34}},numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Modern Physics,en
6,PHYSICS_00013,Consider a small bead with a mass of $m=1 \mat...,"[""\\boxed{1.05 \\times 10^{-33}}"", ""\\boxed{4....","[""numerical"", ""numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Modern Physics,en
7,PHYSICS_00015,To make a flat stone skip across the water sur...,\boxed{\frac{\sqrt{2 g h}}{\tan \theta}},numerical,PHYSICS,High School and Below,train_candidate,,,Mechanics,en
8,PHYSICS_00017,Fishing boats commonly use echo sounders to em...,\boxed{B},mcq,PHYSICS,High School and Below,train_candidate,,,Optics,en
9,PHYSICS_00019,"In a cylinder, a certain amount of ideal gas i...",\boxed{ABD},mcq,PHYSICS,High School and Below,train_candidate,,,Thermodynamics,en


## 7. Sample Records (Random 10)

In [55]:
df.sample(min(10, len(df)))

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
6552,PHYBench_00066,There is a uniform rigid current-carrying circ...,\[\nT_{1}= 2\pi\sqrt{\frac{2M(h^{2}+R^{2})^{7/...,expression,PHYBench,,train_candidate,,,ELECTRICITY,en
1437,UGPhysics_AtomicPhysics_00639,Express the frequency difference of the $\alph...,\boxed{\frac{3}{4}\left(\frac{e^{2}}{4 \pi \va...,expression,UGPhysics,,train_candidate,,,AtomicPhysics,en
6458,OlympiadBench_OE_TO_physics_en_COMP_00207,"a. For large $V_{0}$, the velocity of the posi...",$v \frac{\mathrm{d} \rho}{\mathrm{d} x}+\rho \...,equation,OlympiadBench,,train_candidate,,,OE_TO_physics_en_COMP,en
4813,UGPhysics_StatisticalMechanics_00088,An ideal gas composed of \(N\) indistinguishab...,\[\n\boxed{\mu = -kT\left[\ln \frac{V}{N} + \f...,equation,UGPhysics,,train_candidate,,,StatisticalMechanics,en
3536,UGPhysics_QuantumMechanics_00371,"In atomic units, it is known that there are th...",\boxed{\frac{\mu_{e} e^{4}}{\hbar^{2}}},expression,UGPhysics,,train_candidate,,,QuantumMechanics,en
4296,UGPhysics_Relativity_00127,A stationary atom emits light with an angular ...,\boxed{\omega_{0} \sqrt{\frac{1 + \frac{V}{c}}...,"[""expression"", ""expression""]",UGPhysics,,train_candidate,"None, None",,Relativity,en
6637,SciBench_RL_00051,"Two long, charged, thin-walled, concentric cyl...",2.3,numerical,SciBench_RL,,train_candidate,$10^6 \mathrm{~N} / \mathrm{C} $,,fund,en
2135,UGPhysics_ClassicalMechanics_00034,A rotating spherical planet has a velocity of ...,\boxed{\frac{2 V^{2}}{R}},numerical,UGPhysics,,train_candidate,,,ClassicalMechanics,en
6623,SciBench_RL_00037,Earth's atmosphere is constantly bombarded by ...,122,numerical,SciBench_RL,,train_candidate,$\mathrm{~mA}$,,fund,en
1926,UGPhysics_ClassicalElectromagnetism_00214,A charge is in a condition of stable equilibri...,\boxed{C},mcq,UGPhysics,,train_candidate,,,ClassicalElectromagnetism,en


In [56]:
df

,problem_id,problem,answer,answer_type,source,difficulty,split,unit,tolerance,domain,language
0,PHYSICS_00001,The process of vaporization. At a pressure of ...,\boxed{3.75 \times 10^{4}},numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
1,PHYSICS_00003,Entropy change of hot water. Place 1 kg of wat...,"[""\\boxed{1.01 \\times 10^{3}}"", ""\\boxed{-9.0...","[""numerical"", ""numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Thermodynamics,en
2,PHYSICS_00005,Use white light as the light source to observe...,\boxed{1},numerical,PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
3,PHYSICS_00007,"Under normal brightness, the diameter of the h...","[""\\boxed{2.24 \\times 10^{-4}}"", ""\\boxed{8.9}""]","[""numerical"", ""numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
4,PHYSICS_00009,Example 25.7: If an object is placed within th...,"[""\\boxed{-60}"", ""\\boxed{4}""]","[""numerical"", ""numerical""]",PHYSICS,"Undergraduate (Non-Physics Major),",train_candidate,,,Optics,en
...,...,...,...,...,...,...,...,...,...,...,...
6861,SciBench_RL_00422,An urn contains four colored balls: two orange...,0.2,numerical,SciBench_RL,,train_candidate,,,stat,en
6862,SciBench_RL_00423,"Bowl $B_1$ contains two white chips, bowl $B_2...",0.65625,numerical,SciBench_RL,,train_candidate,,,stat,en
6863,SciBench_RL_00424,Divide a line segment into two parts by select...,0.66666666666,numerical,SciBench_RL,,train_candidate,,,stat,en
6864,SciBench_RL_00425,"In a state lottery, four digits are drawn at r...",0.0024,numerical,SciBench_RL,,train_candidate,,,stat,en


In [57]:
df.iloc[516]['answer']

'["\\\\boxed{V_{\\\\mathrm{eff}}(r) = 0}", "\\\\boxed{V_{\\\\mathrm{eff}}(r) = b^{2} + 2}"]'

## 8. Inspect Individual Records (Interactive)

In [58]:
# View a specific record with nice formatting
idx = 0  # Change this to inspect different rows
print(f"\nRecord #{idx}:")
print("=" * 80)
for col, val in df.iloc[idx].items():
    print(f"{col:30s} : {val}")


Record #0:
problem_id                     : PHYSICS_00001
problem                        : The process of vaporization. At a pressure of $1.013 \times 10^{5} \mathrm{~Pa}$, how much does the internal energy of 1 mol of water increase when it turns into steam at $100^{\circ} \mathrm{C}$? It is known that at this pressure and temperature, the molar volumes of water and steam are $V_{1, \mathrm{~m}}=18.8 \mathrm{~cm}^{3} / \mathrm{mol}$ and $V_{\phi, \mathrm{m}}=3.01 \times 10^{4} \mathrm{~cm}^{3} / \mathrm{mol}$, respectively, and the latent heat of vaporization of water is $L=4.06 \times 10^{4} \mathrm{~J} / \mathrm{mol}$.
answer                         : \boxed{3.75 \times 10^{4}}
answer_type                    : numerical
source                         : PHYSICS
difficulty                     : Undergraduate (Non-Physics Major),
split                          : train_candidate
unit                           : 
tolerance                      : 
domain                         : Thermod

## 9. Summary

In [59]:
print(f"✓ Total records: {len(df):,}")
print(f"✓ Total columns: {len(df.columns)}")
print(f"✓ Columns: {', '.join(df.columns.tolist())}")
print(f"✓ Null values: {df.isnull().sum().sum()}")
print(f"✓ Duplicates: {df.duplicated().sum()}")

✓ Total records: 6,866
✓ Total columns: 11
✓ Columns: problem_id, problem, answer, answer_type, source, difficulty, split, unit, tolerance, domain, language
✓ Null values: 0
✓ Duplicates: 0


In [60]:
# Inspect a specific problem_id
def inspect(problem_id):
    row = df[df.problem_id == problem_id].iloc[0]
    for col, val in row.items():
        print(f"{col:15s}: {val}")

# Inspect a random sample, optionally filtered by source/type
def sample(n=10, source=None, answer_type=None):
    sub = df
    if source:
        sub = sub[sub.source == source]
    if answer_type:
        sub = sub[sub.answer_type.str.contains(answer_type, regex=False)]
    return sub.sample(min(n, len(sub)))[['problem_id','answer','answer_type','problem']].to_string()

# Quick counts
print(df.groupby('source').size())
print(df.answer_type.value_counts().head(20))

source
OlympiadBench     230
PHYBench          100
PHYSICS           805
SciBench_RL       280
UGPhysics        5451
dtype: int64
answer_type
numerical                                                   2698
expression                                                  2030
equation                                                     792
mcq                                                          269
["numerical", "numerical"]                                   268
["expression", "expression"]                                 175
true_false                                                   152
interval                                                      65
["equation", "equation"]                                      63
["numerical", "numerical", "numerical"]                       49
["equation", "expression"]                                    44
["expression", "numerical"]                                   30
["expression", "expression", "expression"]                    26
["numerical",

In [61]:
# These should all be 0 after fix
print("Chinese rows:", df.problem.str.contains(r'[\u4e00-\u9fff]', regex=True).sum())
print("Multi-alt type > answer:", sum(
    1 for _, r in df.iterrows()
    if isinstance(r.answer_type, str) and r.answer_type.startswith('[')
    and not (isinstance(r.answer, str) and r.answer.startswith('['))
))


Chinese rows: 0


Multi-alt type > answer: 676
